В данном ноутбуке мы будем получать через скрепинг вакансии из ххру) 

In [22]:
%pip install -U selenium

Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd
import numpy as np
from time import sleep
import random

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
import logging
from pathlib import Path

In [10]:
logging.basicConfig(level=logging.INFO, filename="hh_scrapping.log", filemode="w")

In [35]:
options = Options()
options.page_load_strategy = 'eager'
driver = webdriver.Chrome(options=options)


# Задаем нужные заголовки через девтулзы. Надо чтобы нас не засекли. 
# Скопировал из того что мой браузер шлет

custom_headers = {
    "headers": {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 YaBrowser/26.3.0.0 Safari/537.36",
        "Accept-Language": "ru,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Refferer": "https://www.yandex.ru/clck/jsredir?from=yandex.ru;suggest;browser&text=",
        "sec-ch-ua-platform": "macOS"
    }
}

driver.execute_cdp_cmd('Network.setExtraHTTPHeaders', custom_headers)

{}

In [12]:
def safeGetElement(element, selector):
    try:
        return element.find_element(By.CSS_SELECTOR, selector)
    except:
        logging.warning(f"Не нашли элемент: {selector}")
        return None

def safeExtractText(element, selector):
    try:
        return getattr(element.find_element(By.CSS_SELECTOR, selector), 'text', '')
    except:
        logging.warning(f"Не достали текст из элемента: {selector}")
        return ''

def safeGetAttr(element, attr):
    try:
        return element.get_attribute(attr)
    except:
        logging.warning(f"Не достали атрибут {attr} из элемента: {element.get_attribute('outerHTML')}")
        return ''

def randSleep():
    sleep(random.uniform(1, 7))


In [13]:
def collect_vacancies():
    page_df = pd.DataFrame({'title': [],'link': [], 'address': [], 'company': [], 'short_description': []})
    vacs = driver.find_elements(By.CSS_SELECTOR, "[data-qa=vacancy-serp__vacancy]")

    for vac in vacs:
        try: 
            link = safeGetAttr(safeGetElement(vac, "[data-qa=serp-item__title]"), 'href')

            duplicates = page_df.loc[page_df['link'] == link]

            if len(duplicates) > 0:
                continue

            title = safeExtractText(vac, "[data-qa=serp-item__title-text]")
            address = safeExtractText(vac, "[data-qa=vacancy-serp__vacancy-address]")
            company = safeExtractText(vac, "[data-qa=vacancy-serp__vacancy-employer-text]")
            short_description_container = safeGetElement(vac, "[data-qa=vacancy-serp__vacancy_snippet_requirement]")

            short_description = ''
            if short_description_container != '':
                short_description = safeExtractText(short_description_container, "span")
            
            # Не пугайтесь это не гпт, я сам это час пытался настроить
            # Мы ищем узел с определенным data-qa, но там внутри пишут обычно частоту вылпаты
            # Поэтому поднимаемся на два уровня вверх и берем первую ноду в ней зп
            xpath_query = ".//*[starts-with(@data-qa, 'vacancy-serp__vacancy-compensation-frequency')]/../../../*[1]"

            try:
                node_with_ruble = vac.find_element(By.XPATH, xpath_query)
            except:
                node_with_ruble = None

            salary = node_with_ruble.text.strip() if node_with_ruble else ''

            new_row = pd.DataFrame({'title': [title], 'link': [link], 'address': [address], 'company': [company], 'short_description': [short_description], 'salary': [salary]})
            page_df = pd.concat([page_df, new_row], ignore_index=True)
        except Exception as e: 
            logging.error(f"Ошибка при обработке конкретной вакансии: {e}")
            continue
    return page_df

In [14]:
def startParsing():
    df_local = pd.DataFrame({'title': [],'link': [], 'address': [], 'company': [], 'short_description': [], 'rich_description': [], 'salary': [], 'skils': []})
    
    try: 
        logging.info(f"Открыли страницу {driver.current_url}")

        while True: 
            next = safeGetElement(driver, "[data-qa=pager-next]")
            ActionChains(driver).scroll_by_amount(0, random.randint(300, 700)).perform()

            randSleep()
            page_df = collect_vacancies()
            df_local = pd.concat([df_local, page_df], ignore_index=True)

            if next:
                ActionChains(driver).move_to_element(next).perform()
                ActionChains(driver).scroll_by_amount(0, 500).perform()

                next.click()
                next = safeGetElement(driver, "[data-qa=pager-next]")
            else:
                logging.info(f"Парсинг закончен, {len(df_local)} вакансий собрано")
                return df_local

    except Exception as e:
        logging.error(f"Ошибка при парсинге: {e}")
        logging.error(f"Текущая страница: {driver.current_url}")
        return df_local

In [ ]:
# Просто с поиском ИТ
driver.get('https://hh.ru/search/vacancy?text=it&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/hh_it.csv', index=False)

In [ ]:
# Только Москва
driver.get('https://hh.ru/search/vacancy?text=it&enable_snippets=true&area=1')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_moscow_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/moscow_it.csv', index=False)

In [ ]:
# Только Питер
driver.get('https://hh.ru/search/vacancy?text=it&enable_snippets=true&area=2')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_spb_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/spb_it.csv', index=False)

In [21]:
# Парсим ui ux дизайнеров
driver.get('https://hh.ru/search/vacancy?text=ui+ux+дизайнер&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_ux_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/ux_it.csv', index=False)

In [22]:
# Парсим веб дизайнеров
driver.get('https://hh.ru/search/vacancy?text=веб+дизайнер&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_web_designer.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/web_designer.csv', index=False)

In [ ]:
# Парсим разрабов
driver.get('https://hh.ru/search/vacancy?text=разработчик&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/devs.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/devs.csv', index=False)

In [32]:
# Парсим разрабов мск
driver.get('https://hh.ru/search/vacancy?text=разработчик&enable_snippets=true&area=1')

logging.basicConfig(level=logging.INFO, filename="./logs/devs_moscow.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/devs_moscow.csv', index=False)

In [33]:
# Парсим разрабов питер
driver.get('https://hh.ru/search/vacancy?text=разработчик&enable_snippets=true&area=2')

logging.basicConfig(level=logging.INFO, filename="./logs/devs_spb.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/devs_spb.csv', index=False)

In [36]:
# Парсим тестировщиков
driver.get('https://hh.ru/search/vacancy?text=тестировщик&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/testers.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/testers.csv', index=False)

In [37]:
# объединяем

folder = Path("./data")

csv_files = folder.glob("*.csv")

df = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

df

,title,link,address,company,short_description,rich_description,salary,skils
0,Разработчик,https://hh.ru/vacancy/131819443?query=%D1%80%D...,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,NaN,NaN,NaN
1,Разработчик,https://hh.ru/vacancy/131986214?query=%D1%80%D...,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",NaN,"от 95 000 ₽ за месяц, на руки",NaN
2,Backend-разработчик,https://hh.ru/vacancy/131274910?query=%D1%80%D...,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,NaN,NaN,NaN
3,Разработчик ЦФТ,https://hh.ru/vacancy/132003073?query=%D1%80%D...,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,NaN,NaN,NaN
4,Разработчик C++,https://hh.ru/vacancy/132001730?query=%D1%80%D...,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
18896,E-mail/CRM маркетолог,https://hh.ru/vacancy/128762563?query=%D1%80%D...,Санкт-Петербург,Devim,"Внедрять ИИ в системные процессы, использовать...",NaN,NaN,NaN
18897,Маркетолог,https://hh.ru/vacancy/131827945?query=%D1%80%D...,Санкт-Петербург,ООО Дримкас,Сосредоточенность и системность. Умение достиг...,NaN,NaN,NaN
18898,Senior ML-инженер,https://hh.ru/vacancy/131724217?query=%D1%80%D...,Санкт-Петербург,ООО Live Typing,Имеешь опыт коммерческой разработки и внедрени...,NaN,"270 000 – 310 000 ₽ за месяц, до вычета налогов",NaN
18899,Инженер по нагрузочному тестированию в банк,https://hh.ru/vacancy/131722980?query=%D1%80%D...,Санкт-Петербург,DCloud,Ответственность и самодисциплина. Умение эффек...,NaN,NaN,NaN
